In [ ]:
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import figure

#import xgboost as xgb
from sklearn import linear_model
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
bronx_2014=pd.read_excel('/Users/liliyang/Desktop/Columbia Classes/Big Data/Datasets/NY City Housing/2014_bronx.xls')


In [ ]:
# Bronx
bronx_frames = [bronx_2019, bronx_2018, bronx_2017, bronx_2016, bronx_2015, bronx_2014]
bronx = pd.concat(bronx_frames)
bronx['SALE DATE\n'] = pd.to_datetime(bronx['SALE DATE\n'])

In [ ]:
# Total data
data_frames = [manhattan, bronx, brooklyn, statenisland, queens]
data = pd.concat(data_frames)

# Some vectorized operations
log = np.vectorize(math.log)
exp = np.vectorize(math.exp)

# Basic filtering
# the following three lines get rid of all the null values
lec = data.isnull().sum(axis=0)/len(data)
lec = lec > 0.2
data = data[data.columns[~lec]]

print(data.isnull().sum(axis=0))
data = data[data['SALE PRICE\n'] >= 100000]
data = data[data['LAND SQUARE FEET\n'] != 0]
data = data[data['GROSS SQUARE FEET\n'] != 0]
data = data[data['TOTAL UNITS\n'] != 0]
data = data[data['RESIDENTIAL UNITS\n'] != 0]
data = data[data['COMMERCIAL UNITS\n'] != 0]
data = data[data['YEAR BUILT\n']>=1750]
data['SALE YEAR'] = pd.DatetimeIndex(data['SALE DATE\n']).year
data['LOG PRICE'] = log(data['SALE PRICE\n'])

In [ ]:
manhattan = manhattan[manhattan['SALE PRICE\n'] >= 100000]
manhattan = manhattan[manhattan['LAND SQUARE FEET\n'] != 0]
manhattan = manhattan[manhattan['GROSS SQUARE FEET\n'] != 0]
manhattan = manhattan[manhattan['TOTAL UNITS\n'] != 0]
manhattan = manhattan[manhattan['RESIDENTIAL UNITS\n'] != 0]
manhattan = manhattan[manhattan['COMMERCIAL UNITS\n'] != 0]
manhattan = manhattan[manhattan['YEAR BUILT\n']>=1750]
manhattan['SALE YEAR'] = pd.DatetimeIndex(manhattan['SALE DATE\n']).year
manhattan['LOG PRICE'] = log(manhattan['SALE PRICE\n'])

In [ ]:
def dummy_variables(df, columns, prefixes):
    df = df.copy()
    for column, prefix in zip(columns, prefixes):
        dummies = pd.get_dummies(df[column], prefix=prefix)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1) 
    return df

## Exploring trends

In [ ]:
# use median log of sale price in the boroughs over time
# not sale price because差距太大图很难分析
l1 = bronx['LOG PRICE'].groupby(bronx['SALE YEAR']).median().plot(color = 'teal', label = 'Bronx')
l1 = manhattan['LOG PRICE'].groupby(manhattan['SALE YEAR']).median().plot(color = 'pink', label = 'Manhattan')
l1.legend(fancybox=True)

In [ ]:
y = np.asarray(data['LOG PRICE'])
sns.distplot(y)

In [ ]:
y_log = log(y)
sns.distplot(y_log)

In [ ]:
borough = data['BOROUGH\n']
borough = borough.astype('category')
sns.boxplot(x=borough, y = y_log)

In [ ]:
# log numbers of residential units vs log price
r_units = data['RESIDENTIAL UNITS\n']
r_units_log = log(r_units)
sns.scatterplot(x = r_units_log, y = y_log, hue = borough)

In [ ]:
taxclass = data['TAX CLASS AT TIME OF SALE\n']
taxclass = taxclass.astype('category')
sns.boxplot(x= taxclass, y =y_log)

In [ ]:
gsf = data['GROSS SQUARE FEET\n']
log_gsf = log(gsf)
sns.scatterplot(x=log_gsf, y = y_log, hue = borough)

## Modeling

In [ ]:
columns_selected = ['BOROUGH\n','TAX CLASS AT TIME OF SALE\n', 'SALE PRICE\n', 'GROSS SQUARE FEET\n', 'COMMERCIAL UNITS\n','RESIDENTIAL UNITS\n', 'YEAR BUILT\n']
categorical_variables = ['BOROUGH\n', 'TAX CLASS AT TIME OF SALE\n']
model_data = data[columns_selected]
y = model_data['SALE PRICE\n']
y_log = log(y)
model_data = model_data.drop(['SALE PRICE\n'], axis=1)
model_data['GROSS SQUARE FEET\n'] = log(model_data['GROSS SQUARE FEET\n'])
model_data['RESIDENTIAL UNITS\n'] = log(model_data['RESIDENTIAL UNITS\n'])
model_data['COMMERCIAL UNITS\n'] = log(model_data['COMMERCIAL UNITS\n'])
model_data = dummy_variables(model_data, categorical_variables, ['BO', 'TC', 'ZC'])
model_data = model_data.fillna(model_data.mean())

In [ ]:
# linear regression
X_train, X_test, y_train, y_test = train_test_split(model_data, y_log, test_size=0.5, random_state=0)
regression = linear_model.LinearRegression()
regression.fit(X_train, y_train)
y_pred = regression.predict(X_test)
#print('Coefficients: \n', regr.coef_)
print('Mean squared error: %.2f'
      % mean_squared_error(y_test,y_pred))
print('Coefficient of determination: %.2f'
      % r2_score((y_test), (y_pred)))

In [ ]:
# lasso regression
X_train, X_test, y_train, y_test = train_test_split(model_data, y_log,test_size=0.5, random_state=0)
alpha=0.00099
lasso = Lasso(alpha=alpha,max_iter=50000)
lasso.fit(X_train, y_train)
y_pred=lasso.predict(X_test)
print('Mean squared error: %.2f'
      % (mean_squared_error((y_test),(y_pred))))
print('Coefficient of determination: %.2f'
      % r2_score(y_test, y_pred))

In [ ]:
# ridge regression
X_train, X_test, y_train, y_test = train_test_split(model_data, y_log,test_size=0.5, random_state=0)
ridge = Ridge(alpha=0.01, normalize=True)
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)
print('Mean squared error: %.2f'
      % (mean_squared_error((y_test), (y_pred))))
print('Coefficient of determination: %.2f'
      % r2_score(y_test, y_pred))

In [ ]:
# extreme gradient boosting
X_train, X_test, y_train, y_test = train_test_split(model_data, y_log, train_size=0.7, random_state=123)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=123)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

In [ ]:
%%capture
params = {'learning_rate': 0.001, 'max_depth': 6, 'lambda': 0.01}
model = xgb.train(params, dtrain, num_boost_round=10000, evals=[(dval, 'eval')], early_stopping_rounds=10);

In [ ]:
y_true = np.array(y_test)
y_pred = model.predict(dtest)

In [ ]:
print('Mean squared error: %.2f'
      % (mean_squared_error((y_test), (y_pred))))
print('Coefficient of determination: %.2f'
      % r2_score(y_test, y_pred))